In [ ]:
import kagglehub
import pandas as pd
import os

# 1. Baixa o dataset e descobre o caminho real onde ele foi salvo no seu PC
path = kagglehub.dataset_download("namespaiva/base-varejo")

# 2. Junta o caminho da pasta com o nome do seu arquivo CSV

full_path = os.path.join(path, "Base Varejo.csv")

# 3. Lê o arquivo normalmente usando o Pandas
df = pd.read_csv(full_path, encoding='utf-8', sep=';')

#Criando dicionario para mapear os códigos de estado civil para seus significados de acordo com a descrição do dataset
dic_estado_civil = {1: "Casado/UE", 2: "Divorciado", 3: "Separado", 4: "Solteiro", 5: "Viúvo", 6: "Viúvo(a)"}

print("\nInformações do DataFrame:")
print(df.shape)
print(df.info())
print()
print("Primeiros 5 registros:")
print(df.head())


# limpeza

### Limpando e convertendo nomes e tipos de Colunas

- Removendo colunas em branco "Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13".
- Renomeando colunas de acordo com a descrição do dataset
- Convetendo data para datetime

In [ ]:

# remove as colunas extras que não possuem nome
df = df.drop(columns=["Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13"])

# renomeia as colunas para nomes mais amigáveis
df = df.rename(columns={"CO_ID": "N_fiscal", "CL_ID": "ID_Cliente", "CL_GENERO": "Genero", "CL_EC": "Estado_Civil", "CL_FHL": "Qtd_Filhos", "CL_SEG": "Seg_Economica", "PR_ID": "ID_Produto", "PR_CAT": "Cat_Produto",
                         "PR_NOME": "Nome_Produto"})
#converte a coluna "DATA" para o formato de datetime do Pandas
df["DATA"] = pd.to_datetime(
    df["DATA"],
    format="%d/%m/%Y",
    errors="coerce")


## Duplicatas

In [ ]:
# Identificando duplicatas
dup_mask = df.duplicated()
print(f"Total de duplicatas: {dup_mask.sum()}")
print()

# # Exemplo das duplicatas encontradas
print("Primeiras 4 duplicatas:")
print(df[dup_mask].head(4).to_string()) 
#removendo as duplicatas
df_limpo = df.drop_duplicates()
print(f"Tamanho original: {len(df)} - Tamanho após remoção de duplicatas: {len(df_limpo)}, foram removidas {(len(df) - len(df_limpo))/len(df)*100:.2f}% de linhas totais que eram duplicadas.")


In [ ]:
print(df_limpo.info())
print(df_limpo.head())

### Analisando dados

In [ ]:
dic_estado_civil = {1: "Casado/UE", 2: "Divorciado", 3: "Separado", 4: "Solteiro", 5: "Viúvo", 6: "Viúvo(a)"}
print(df_limpo["Estado_Civil"].map(dic_estado_civil).value_counts())

In [ ]:
colunas_categoricas = ["Genero", "Estado_Civil", "Qtd_Filhos", "Seg_Economica", "Cat_Produto"]
for col in colunas_categoricas:
    if col == "Estado_Civil":
        print(f"{(df_limpo[col].map(dic_estado_civil).value_counts()/len(df_limpo[col])*100).round(2)}%")
    else:
        print(f"\nContagem de valores para a coluna '{col}':")
        print(f"{(df_limpo[col].value_counts()/len(df_limpo[col])*100).round(2)}%")
        print("-"*20)


Identificado Valor #N/D nas categorias do produto, verificando se existe algum padrão.

In [ ]:
df_analise = df_limpo.query("Cat_Produto == '#N/D'")
print(df_analise.head())
print("-"*20)
print("Os valores de ID_produto com #N/D na coluna 'Cat_Produto' são:")
print(df_analise["ID_Produto"].unique())

**Decisão**

Já que foi identificado que apenas um produto ficou com o termo #N/D foi decidido trata-lo como desconhecido para manter os demais dados.

In [ ]:
df_limpo["Cat_Produto"] = df_limpo["Cat_Produto"].replace("#N/D", "Desconhecido")
df_limpo["Nome_Produto"] = df_limpo["Nome_Produto"].replace("#N/D", "Desconhecido")

In [ ]:
print((df_limpo["Cat_Produto"].value_counts()/len(df["Cat_Produto"])*100).round(2))
print("-"*20)
print((df_limpo["Nome_Produto"].value_counts()/len(df["Nome_Produto"])*100).round(2))

# Analise

## Analise Quantidade de filhos

### Estatisticas

In [114]:
print("=" * 10 + " Análise estatística da coluna 'Qtd_Filhos' " + "=" * 10)
print(f"Mediana: {df_limpo['Qtd_Filhos'].median()}")
print(f"Média: {df_limpo['Qtd_Filhos'].mean()}")
print(f"Moda: {df_limpo['Qtd_Filhos'].mode().iloc[0] if not df_limpo['Qtd_Filhos'].mode().empty else 'Nenhuma'}")
print(f"Valor Máximo: {df_limpo['Qtd_Filhos'].max()}")
print(f"Valor Mínimo: {df_limpo['Qtd_Filhos'].min()}")
print()
print("Contagem de valores 'Qtd_Filhos':")
contagem_qtd_filhos = pd.DataFrame([df_limpo["Qtd_Filhos"].value_counts(), (df_limpo["Qtd_Filhos"].value_counts()/len(df_limpo["Qtd_Filhos"])*100).round(2)])
# 2. Inverte as linhas e colunas usando o .T
# Também vamos dar nomes claros para as colunas para ficar organizado
contagem_qtd_filhos = contagem_qtd_filhos.T
contagem_qtd_filhos.columns = ['Quantidade', 'Percentual']

# 3. Adiciona o símbolo de % na coluna de Percentual
contagem_qtd_filhos['Percentual'] = contagem_qtd_filhos['Percentual'].map('{:.2f}%'.format)
contagem_qtd_filhos['Quantidade'] = contagem_qtd_filhos['Quantidade'].astype(int)
contagem_qtd_filhos = contagem_qtd_filhos.sort_index(ascending=True)

# Visualizar o resultado
print(contagem_qtd_filhos)


========== Análise estatística da coluna 'Qtd_Filhos' ==========
Mediana: 0.0
Média: 1.1460487260838206
Moda: 0
Valor Máximo: 4
Valor Mínimo: 0

Contagem de valores 'Qtd_Filhos':
            Quantidade Percentual
Qtd_Filhos                       
0               384986     52.49%
1                90845     12.39%
2                94168     12.84%
3                92407     12.60%
4                71041      9.69%


**Insights**
- Há compradores sem filhos e esses são a maioria (+50%).
- O maior numero de filhos por comprador é 4 e essas são a minoria.

### Agrupamentos